# 14 — Build Country-Year Feature Matrix

Assembles the master **country-year panel** used by notebooks 15–17 for instability prediction.
Reads cleaned parquets written by notebooks 01–13 from ADLS, joins them on a common ISO3 key,
and codes five binary outcome labels (all forward-shifted one year).

## Panel dimensions
- **Unit:** country-year (~167 countries × 25 years = ~4,000 rows)
- **Feature window:** 2000–2024
- **Training labels available for:** 2000–2023 (label at year *t* requires data from year *t+1*)

## Outcome labels (binary, forward-shifted +1 year)

| Label | Ground truth | Coding rule |
|---|---|---|
| `civil_war_onset` | UCDP-GED | First year of state-based conflict after ≥2-year peace spell |
| `coup_attempt` | Powell-Thyne | Any coup attempt (success or failure) in year *t+1* |
| `regime_backsliding` | V-Dem | `v2x_libdem` drops ≥0.05 in year *t+1*, or regime transitions to closed autocracy |
| `mass_unrest_onset` | ACLED | Annual protest+riot events exceed country 90th percentile in year *t+1* |
| `humanitarian_crisis_onset` | FEWS NET | Country enters IPC Phase ≥4 in year *t+1*, given Phase ≤3 at *t* (~39 countries) |

## Sources joined
Monthly sources (ACLED, GDELT, FAO food prices) are **aggregated to annual statistics**
before joining — the panel unit is country-year throughout.

## ADLS output
```
processed/feature_matrix/{RUN_DATE}/feature_matrix.parquet   — full feature panel
processed/feature_matrix/{RUN_DATE}/labels.parquet           — iso3 + year + 5 outcome columns
```

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER  (default: 'data')
```

In [ ]:
import os
import warnings
import re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from azure.identity import DefaultAzureCredential
import adlfs

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

PANEL_START_YEAR  = 2000
PANEL_END_YEAR    = 2024
LABEL_HORIZON     = 1   # predict outcomes 1 year ahead

# Temporal split boundaries (year-inclusive)
TRAIN_END_YEAR    = 2018
VAL_END_YEAR      = 2021
# Test: 2022–2024

# ADLS prefixes for each raw source (latest date partition selected automatically)
RAW_PREFIXES = {
    "acled_monthly":    "raw/acled/monthly_agg",
    "wdi":              "raw/world_bank/wdi",
    "wgi":              "raw/world_bank/wgi",
    "vdem":             "raw/vdem",
    "polity5":          "raw/polity5",
    "ucdp_ged_cy":      "raw/ucdp_ged",
    "powell_thyne":     "raw/powell_thyne",
    "pitf":             "raw/pitf",
    "fsi":              "raw/fsi",
    "unhcr":            "raw/unhcr",
    "undp_hdi":         "raw/undp_hdi",
    "gdelt":            "raw/gdelt",
    "fao_ffpi":         "raw/fao/ffpi",
    "fao_cpi":          "raw/fao/country_cpi",
    "sipri":            "raw/sipri",
    "prio_grid_cy":     "raw/prio_grid",
    "archigos_cy":      "raw/archigos",
    "alc_cy":           "raw/alc",
    "cnts":             "raw/cnts",
    "nelda":            "raw/nelda",
}

print(f"Run date       : {RUN_DATE}")
print(f"Panel years    : {PANEL_START_YEAR}–{PANEL_END_YEAR}")
print(f"Temporal split : train ≤{TRAIN_END_YEAR} | val ≤{VAL_END_YEAR} | test >={VAL_END_YEAR+1}")

## ADLS helpers

In [ ]:
credential     = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

def read_latest_parquet(prefix: str) -> pd.DataFrame | None:
    """
    List all date-partitioned subdirectories under `prefix` and read
    the parquet from the lexicographically latest one (most recent run date).
    Returns None if no parquet files are found.
    """
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    full_prefix = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = fs.ls(full_prefix, detail=False)
    except FileNotFoundError:
        print(f"  WARNING: prefix not found: {full_prefix}")
        return None

    # Keep only date-partition directories (8-digit names)
    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)],
        reverse=True,
    )
    if not date_dirs:
        print(f"  WARNING: no date partitions found under {full_prefix}")
        return None

    latest_dir = date_dirs[0]
    parquet_files = [
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + f.replace(f"{ADLS_CONTAINER}/", "", 1)
        for f in fs.glob(f"{latest_dir}/*.parquet")
    ]
    if not parquet_files:
        print(f"  WARNING: no .parquet files in {latest_dir}")
        return None

    dfs = [pd.read_parquet(p, storage_options=storage_options) for p in parquet_files]
    df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
    print(f"  Loaded {len(df):,} rows from {latest_dir}")
    return df

## Country crosswalk

Loads the static crosswalk from `data/country_crosswalk.csv` (committed to the repo).
Provides lookup dicts for each join-key type → ISO3.

In [ ]:
# The crosswalk CSV lives alongside the notebooks in the repo
_crosswalk_path = Path("../data/country_crosswalk.csv")
if not _crosswalk_path.exists():
    _crosswalk_path = Path("data/country_crosswalk.csv")

df_cw = pd.read_csv(_crosswalk_path, dtype=str)
df_cw["cow_numeric"]  = pd.to_numeric(df_cw["cow_numeric"],  errors="coerce")
df_cw["gw_numeric"]   = pd.to_numeric(df_cw["gw_numeric"],   errors="coerce")
df_cw["iso_numeric"]  = pd.to_numeric(df_cw["iso_numeric"],  errors="coerce")
df_cw["fews_monitored"] = df_cw["fews_monitored"].astype(int)

# Lookup dicts  →  ISO3
cow_to_iso3  = dict(zip(df_cw["cow_numeric"],  df_cw["iso3"]))
gw_to_iso3   = dict(zip(df_cw["gw_numeric"],   df_cw["iso3"]))
iso2_to_iso3 = dict(zip(df_cw["iso2"],         df_cw["iso3"]))
inum_to_iso3 = dict(zip(df_cw["iso_numeric"],  df_cw["iso3"]))
cameo_to_iso3 = dict(zip(df_cw["cameo2"],      df_cw["iso3"]))

# Name normalisation helper for string-based sources (SIPRI, FSI, UNDP)
_NAME_OVERRIDES = {
    "cote d'ivoire":              "CIV",
    "ivory coast":                "CIV",
    "iran, islamic rep.":         "IRN",
    "iran (islamic republic of)": "IRN",
    "republic of korea":          "KOR",
    "korea, rep.":                "KOR",
    "korea, south":               "KOR",
    "democratic republic of the congo": "COD",
    "congo, dem. rep.":           "COD",
    "dr congo":                   "COD",
    "congo, rep.":                "COG",
    "syrian arab republic":       "SYR",
    "bolivia (plurinational state of)": "BOL",
    "tanzania, united republic of":    "TZA",
    "united republic of tanzania":     "TZA",
    "viet nam":                   "VNM",
    "lao pdr":                    "LAO",
    "lao people's democratic republic": "LAO",
    "kyrgyz republic":            "KGZ",
    "czech republic":             "CZE",
    "slovak republic":            "SVK",
    "turkiye":                    "TUR",
    "turkey":                     "TUR",
    "egypt, arab rep.":           "EGY",
    "gambia, the":                "GMB",
    "bahamas, the":               "BHS",
    "yemen, rep.":                "YEM",
    "venezuela, rb":              "VEN",
    "micronesia, fed. sts.":      "FSM",
    "russian federation":         "RUS",
    "timor-leste":                "TLS",
    "east timor":                 "TLS",
    "cabo verde":                 "CPV",
    "eswatini":                   "SWZ",
    "swaziland":                  "SWZ",
    "north macedonia":            "MKD",
    "macedonia, former yugoslav republic of": "MKD",
}
_name_to_iso3 = dict(zip(
    df_cw["country_name_canonical"].str.lower().str.strip(),
    df_cw["iso3"]
))

def name_to_iso3(name: str) -> str | None:
    if not isinstance(name, str):
        return None
    key = re.sub(r"[^a-z0-9 ]", "", name.lower().strip())
    key_punc = name.lower().strip()
    return (
        _NAME_OVERRIDES.get(key_punc)
        or _name_to_iso3.get(key_punc)
        or _name_to_iso3.get(key)
    )

FEWS_COUNTRIES = set(df_cw.loc[df_cw["fews_monitored"] == 1, "iso3"])

print(f"Crosswalk loaded: {len(df_cw)} countries")
print(f"FEWS-monitored  : {len(FEWS_COUNTRIES)} countries")